In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("superpotato9/dalle-recognition-dataset")

print("Path to dataset files:", path)

In [ ]:
print(path)

In [ ]:
import os

path = "/kaggle/input/dalle-recognition-dataset"
for file in os.listdir(path):
    print(file)

In [ ]:
"""
import os
import cv2
import numpy as np
from glob import glob
from tqdm import tqdm
from scipy.stats import entropy
from scipy.fftpack import fft2, fftshift
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from pathlib import Path

#############################################
# -------- Feature Extraction -------------
#############################################

path = "/kaggle/input/dalle-recognition-dataset"

def shannon_entropy(img):
    hist = np.histogram(img.flatten(), bins=256, range=(0,256))[0]
    prob = hist / np.sum(hist)
    prob = prob[prob > 0]
    return entropy(prob)

def local_entropy(img, ksize=9):
    from skimage.filters.rank import entropy as local_ent
    from skimage.morphology import disk
    img_uint = (img * 255).astype(np.uint8)
    ent = local_ent(img_uint, disk(ksize))
    return np.mean(ent)

def laplacian_variance(img):
    return cv2.Laplacian(img, cv2.CV_64F).var()

def gradient_entropy(img):
    gx = cv2.Sobel(img, cv2.CV_64F, 1, 0, ksize=3)
    gy = cv2.Sobel(img, cv2.CV_64F, 0, 1, ksize=3)
    grad_mag = np.sqrt(gx**2 + gy**2)
    return shannon_entropy(grad_mag)

def high_freq_energy(img):
    f = fft2(img)
    fshift = fftshift(f)
    magnitude = np.abs(fshift)
    h, w = magnitude.shape
    center = magnitude[h//4:3*h//4, w//4:3*w//4]
    total_energy = np.sum(magnitude)
    center_energy = np.sum(center)
    return (total_energy - center_energy) / total_energy

def extract_features(image_path):
    img = cv2.imread(image_path)
    if img is None:
        print(f"Warning: could not read {image_path}")
        return None  # skip this image

    img = cv2.resize(img, (256, 256))
    img = img / 255.0
    gray = cv2.cvtColor((img*255).astype(np.uint8), cv2.COLOR_BGR2GRAY)
    gray = gray / 255.0

    features = []
    features.append(shannon_entropy(gray))
    features.append(local_entropy(gray))
    features.append(gradient_entropy(gray))
    features.append(laplacian_variance(gray))
    features.append(high_freq_energy(gray))
    for c in range(3):
        channel = img[:,:,c]
        features.append(np.mean(channel))
        features.append(np.std(channel))

    return np.array(features, dtype=np.float32)

#############################################
# -------- Precompute Features -------------
#############################################

base_path = Path(path)
real_files = list((base_path / "real").rglob("*.*"))
fake_files = list((base_path / "fakeV2").rglob("*.*"))

all_files = [(str(f), 0) for f in real_files] + [(str(f), 1) for f in fake_files]

features_list = []
labels_list = []

print("Precomputing features for all images...")
for path, label in tqdm(all_files):
    feat = extract_features(path)
    if feat is not None:
        features_list.append(feat)
        labels_list.append(label)

features_array = np.array(features_list, dtype=np.float32)
labels_array = np.array(labels_list, dtype=np.int64)

# Save features for future runs (optional)
np.save("features.npy", features_array)
np.save("labels.npy", labels_array)

#############################################
# -------- Dataset -------------------------
#############################################

class StatsDataset(Dataset):
    def __init__(self, features, labels, indices):
        self.features = features[indices]
        self.labels = labels[indices]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        x = torch.tensor(self.features[idx], dtype=torch.float32)
        y = torch.tensor(self.labels[idx], dtype=torch.float32)
        return x, y

# Split indices for 60/20/20 train/val/test
indices = np.arange(len(labels_array))
train_idx, temp_idx = train_test_split(indices, test_size=0.4, random_state=42)
val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, random_state=42)

train_dataset = StatsDataset(features_array, labels_array, train_idx)
valid_dataset = StatsDataset(features_array, labels_array, val_idx)
test_dataset  = StatsDataset(features_array, labels_array, test_idx)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=16)
test_loader  = DataLoader(test_dataset, batch_size=16)

input_dim = features_array.shape[1]

#############################################
# -------- Model ---------------------------
#############################################

class StatsClassifier(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.BatchNorm1d(32),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)

#############################################
# -------- Training ------------------------
#############################################

def train_model(train_loader, val_loader, test_loader, input_dim, epochs=20, lr=0.001):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Using device:", device)

    model = StatsClassifier(input_dim).to(device)
    criterion = nn.BCELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    train_losses = []
    val_losses = []
    val_accuracies = []

    for epoch in range(epochs):
        # ---------- TRAIN ----------
        model.train()
        running_train_loss = 0
        for x, y in train_loader:
            x = x.to(device)
            y = y.float().unsqueeze(1).to(device)

            optimizer.zero_grad()
            preds = model(x)
            loss = criterion(preds, y)
            loss.backward()
            optimizer.step()

            running_train_loss += loss.item() * x.size(0)

        train_loss = running_train_loss / len(train_loader.dataset)
        train_losses.append(train_loss)

        # ---------- VALIDATION ----------
        model.eval()
        running_val_loss = 0
        all_preds, all_labels = [], []

        with torch.no_grad():
            for x, y in val_loader:
                x = x.to(device)
                y = y.float().unsqueeze(1).to(device)

                preds = model(x)
                loss = criterion(preds, y)
                running_val_loss += loss.item() * x.size(0)

                all_preds.extend((preds > 0.5).int().cpu().squeeze().tolist())
                all_labels.extend(y.cpu().squeeze().tolist())

        val_loss = running_val_loss / len(val_loader.dataset)
        val_acc = accuracy_score(all_labels, all_preds)

        val_losses.append(val_loss)
        val_accuracies.append(val_acc)

        print(f"Epoch {epoch+1}/{epochs} - "
              f"Train Loss: {train_loss:.4f} - "
              f"Val Loss: {val_loss:.4f} - "
              f"Val Acc: {val_acc:.4f}")

    # ---------- TEST SET EVALUATION ----------
    model.eval()
    all_preds, all_labels = [], []
    running_test_loss = 0

    with torch.no_grad():
        for x, y in test_loader:
            x = x.to(device)
            y = y.float().unsqueeze(1).to(device)

            preds = model(x)
            loss = criterion(preds, y)
            running_test_loss += loss.item() * x.size(0)

            all_preds.extend((preds > 0.5).int().cpu().squeeze().tolist())
            all_labels.extend(y.cpu().squeeze().tolist())

    test_loss = running_test_loss / len(test_loader.dataset)
    test_acc = accuracy_score(all_labels, all_preds)

    print(f"\nFinal Test Loss: {test_loss:.4f} - Test Accuracy: {test_acc:.4f}")

    return model, train_losses, val_losses, val_accuracies

#############################################
# -------- Run Training --------------------
#############################################

model, train_losses, val_losses, val_accuracies = train_model(
    train_loader, valid_loader, test_loader, input_dim, epochs=20
)

torch.save(model.state_dict(), "stat_detector.pth")
print("Model saved as stat_detector.pth")
"""

In [ ]:
import os
import cv2
import numpy as np
from glob import glob
from tqdm import tqdm
from scipy.stats import entropy
from scipy.fftpack import fft2, fftshift
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from pathlib import Path
import csv

#############################################
# -------- Feature Extraction -------------
#############################################

path = "/kaggle/input/dalle-recognition-dataset"

def shannon_entropy(img):
    hist = np.histogram(img.flatten(), bins=256, range=(0,256))[0]
    prob = hist / np.sum(hist)
    prob = prob[prob > 0]
    return entropy(prob)

def local_entropy(img, ksize=9):
    from skimage.filters.rank import entropy as local_ent
    from skimage.morphology import disk
    img_uint = (img * 255).astype(np.uint8)
    ent = local_ent(img_uint, disk(ksize))
    return np.mean(ent)

def laplacian_variance(img):
    return cv2.Laplacian(img, cv2.CV_64F).var()

def gradient_entropy(img):
    gx = cv2.Sobel(img, cv2.CV_64F, 1, 0, ksize=3)
    gy = cv2.Sobel(img, cv2.CV_64F, 0, 1, ksize=3)
    grad_mag = np.sqrt(gx**2 + gy**2)
    return shannon_entropy(grad_mag)

def high_freq_energy(img):
    f = fft2(img)
    fshift = fftshift(f)
    magnitude = np.abs(fshift)
    h, w = magnitude.shape
    center = magnitude[h//4:3*h//4, w//4:3*w//4]
    total_energy = np.sum(magnitude)
    center_energy = np.sum(center)
    return (total_energy - center_energy) / total_energy

def extract_features(image_path):
    img = cv2.imread(image_path)
    if img is None:
        print(f"Warning: could not read {image_path}")
        return None

    img = cv2.resize(img, (256, 256))
    img = img / 255.0
    gray = cv2.cvtColor((img*255).astype(np.uint8), cv2.COLOR_BGR2GRAY)
    gray = gray / 255.0

    features = [
        shannon_entropy(gray),
        local_entropy(gray),
        gradient_entropy(gray),
        laplacian_variance(gray),
        high_freq_energy(gray)
    ]

    for c in range(3):
        channel = img[:,:,c]
        features.append(np.mean(channel))
        features.append(np.std(channel))

    return np.array(features, dtype=np.float32)

#############################################
# -------- Precompute Features -------------
#############################################

base_path = Path(path)
real_files = list((base_path / "real").rglob("*.*"))
fake_files = list((base_path / "fakeV2").rglob("*.*"))

all_files = [(str(f), 0) for f in real_files] + [(str(f), 1) for f in fake_files]

features_list = []
labels_list = []

print("Precomputing features for all images...")
for path, label in tqdm(all_files):
    feat = extract_features(path)
    if feat is not None:
        features_list.append(feat)
        labels_list.append(label)

features_array = np.array(features_list, dtype=np.float32)
labels_array = np.array(labels_list, dtype=np.int64)

# Optional: save features for future use
np.save("features.npy", features_array)
np.save("labels.npy", labels_array)

#############################################
# -------- Dataset -------------------------
#############################################

class StatsDataset(Dataset):
    def __init__(self, features, labels, indices):
        self.features = features[indices]
        self.labels = labels[indices]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        x = torch.tensor(self.features[idx], dtype=torch.float32)
        y = torch.tensor(self.labels[idx], dtype=torch.float32)
        return x, y

# Train/val/test split with reproducibility
np.random.seed(42)
indices = np.arange(len(labels_array))
train_idx, temp_idx = train_test_split(indices, test_size=0.4, random_state=42)
val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, random_state=42)

train_dataset = StatsDataset(features_array, labels_array, train_idx)
valid_dataset = StatsDataset(features_array, labels_array, val_idx)
test_dataset  = StatsDataset(features_array, labels_array, test_idx)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=16)
test_loader  = DataLoader(test_dataset, batch_size=16)

input_dim = features_array.shape[1]

#############################################
# -------- Model ---------------------------
#############################################

class StatsClassifier(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.BatchNorm1d(32),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)

#############################################
# -------- Training ------------------------
#############################################

def train_model(train_loader, val_loader, test_loader, input_dim, epochs=20, lr=0.001):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Using device:", device)

    model = StatsClassifier(input_dim).to(device)
    criterion = nn.BCELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    train_losses, val_losses, val_accuracies = [], [], []

    # -------- TRAINING LOOP --------
    for epoch in range(epochs):
        model.train()
        running_train_loss = 0
        for x, y in train_loader:
            x = x.to(device)
            y = y.float().unsqueeze(1).to(device)

            optimizer.zero_grad()
            preds = model(x)
            loss = criterion(preds, y)
            loss.backward()
            optimizer.step()

            running_train_loss += loss.item() * x.size(0)

        train_loss = running_train_loss / len(train_loader.dataset)
        train_losses.append(train_loss)

        # VALIDATION
        model.eval()
        running_val_loss = 0
        all_preds, all_labels = [], []
        with torch.no_grad():
            for x, y in val_loader:
                x = x.to(device)
                y = y.float().unsqueeze(1).to(device)

                preds = model(x)
                loss = criterion(preds, y)
                running_val_loss += loss.item() * x.size(0)

                all_preds.extend((preds > 0.5).int().cpu().squeeze().tolist())
                all_labels.extend(y.cpu().squeeze().tolist())

        val_loss = running_val_loss / len(val_loader.dataset)
        val_acc = accuracy_score(all_labels, all_preds)
        val_losses.append(val_loss)
        val_accuracies.append(val_acc)

        print(f"Epoch {epoch+1}/{epochs} - Train Loss: {train_loss:.4f} - "
              f"Val Loss: {val_loss:.4f} - Val Acc: {val_acc:.4f}")

    # SAVE TRAIN/VAL LOG
    with open("train_val_log.csv", "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["Epoch", "Train_Loss", "Val_Loss", "Val_Accuracy"])
        for i in range(epochs):
            writer.writerow([i+1, train_losses[i], val_losses[i], val_accuracies[i]])
    print("Training logs saved to train_val_log.csv")

    # -------- TEST EVALUATION --------
    model.eval()
    all_preds, all_labels = [], []
    running_test_loss = 0
    misclassified_files = []

    test_files = np.array([all_files[i][0] for i in test_idx])

    with torch.no_grad():
        for batch_idx, (x, y) in enumerate(test_loader):
            x = x.to(device)
            y = y.float().unsqueeze(1).to(device)

            preds = model(x)
            loss = criterion(preds, y)
            running_test_loss += loss.item() * x.size(0)

            batch_probs = preds.cpu().squeeze().tolist()
            batch_labels = y.cpu().squeeze().tolist()
            batch_preds = [1 if p > 0.5 else 0 for p in batch_probs]

            all_preds.extend(batch_preds)
            all_labels.extend(batch_labels)

            # Collect misclassified filenames with confidence
            start_idx = batch_idx * test_loader.batch_size
            for i, (pred, label, prob) in enumerate(zip(batch_preds, batch_labels, batch_probs)):
                if pred != int(label):
                    misclassified_files.append((test_files[start_idx + i], int(label), pred, float(prob)))

    test_loss = running_test_loss / len(test_loader.dataset)
    test_acc = accuracy_score(all_labels, all_preds)
    print(f"\nFinal Test Loss: {test_loss:.4f} - Test Accuracy: {test_acc:.4f}")

    # SAVE MISCLASSIFIED IMAGES
    with open("misclassified_test_images.csv", "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["filepath", "true_label", "predicted_label", "confidence"])
        for fname, true_label, pred_label, conf in misclassified_files:
            writer.writerow([fname, true_label, pred_label, conf])
    print(f"{len(misclassified_files)} misclassified test images saved to misclassified_test_images.csv")

    return model, train_losses, val_losses, val_accuracies

#############################################
# -------- Run Training --------------------
#############################################

model, train_losses, val_losses, val_accuracies = train_model(
    train_loader, valid_loader, test_loader, input_dim, epochs=20
)

torch.save(model.state_dict(), "stat_detector.pth")
print("Model saved as stat_detector.pth")

In [ ]:
import matplotlib.pyplot as plt

epochs = len(train_losses)
plt.figure(figsize=(8,5))
plt.plot(range(1, epochs+1), train_losses, label="Train Loss", marker='o')
plt.plot(range(1, epochs+1), val_losses, label="Validation Loss", marker='s')
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import numpy as np
from scipy.stats import ttest_ind
import matplotlib.pyplot as plt
import seaborn as sns

# Split features by class
real_features = features_array[labels_array == 0]
fake_features = features_array[labels_array == 1]

# Function to compute Cohen's d effect size
def cohens_d(x, y):
    nx, ny = len(x), len(y)
    pooled_std = np.sqrt(((nx - 1) * np.var(x, ddof=1) + (ny - 1) * np.var(y, ddof=1)) / (nx + ny - 2))
    return (np.mean(x) - np.mean(y)) / pooled_std

feature_names = [
    "Shannon Entropy",
    "Local Entropy",
    "Gradient Entropy",
    "Laplacian Variance",
    "High-Freq Energy",
    "R_mean", "R_std",
    "G_mean", "G_std",
    "B_mean", "B_std"
]

results = []

print("Feature Analysis: p-value (t-test) & Effect Size (Cohen's d)\n")
for i, fname in enumerate(feature_names):
    real_feat = real_features[:, i]
    fake_feat = fake_features[:, i]

    # Two-sample t-test
    t_stat, p_value = ttest_ind(real_feat, fake_feat, equal_var=False)

    # Cohen's d
    d = cohens_d(real_feat, fake_feat)

    results.append((fname, p_value, d))
    print(f"{fname:<20}: p-value = {p_value:.4e}, Cohen's d = {d:.3f}")

# -----------------------------
# Sort by absolute Cohen's d
results_sorted = sorted(results, key=lambda x: abs(x[2]), reverse=True)
top_features = [r[0] for r in results_sorted[:5]]  # Top 5 most discriminative

# -----------------------------
# Plot Cohen's d
plt.figure(figsize=(10,5))
sns.barplot(x=[r[0] for r in results_sorted], y=[r[2] for r in results_sorted], palette="viridis")
plt.xticks(rotation=45, ha='right')
plt.ylabel("Cohen's d (Effect Size)")
plt.title("Feature Effect Sizes: Real vs AI Images")
plt.axhline(0, color='black', linestyle='--')
plt.tight_layout()
plt.show()

# -----------------------------
# Plot distributions for top features
for fname in top_features:
    idx = feature_names.index(fname)
    plt.figure(figsize=(6,4))
    sns.kdeplot(real_features[:, idx], label="Real", fill=True, alpha=0.5)
    sns.kdeplot(fake_features[:, idx], label="AI", fill=True, alpha=0.5)
    plt.title(f"{fname} Distribution: Real vs AI")
    plt.xlabel("Feature Value")
    plt.ylabel("Density")
    plt.legend()
    plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, average_precision_score, confusion_matrix, roc_curve
import seaborn as sns

# -----------------------------
# 1️⃣ Convert test data to arrays
# -----------------------------
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.eval()
all_preds, all_labels, all_probs = [], [], []

with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        y = y.float().unsqueeze(1).to(device)

        preds = model(x)
        probs = preds.cpu().squeeze().numpy()
        labels = y.cpu().squeeze().numpy()
        pred_labels = (probs > 0.5).astype(int)

        all_probs.extend(probs)
        all_labels.extend(labels)
        all_preds.extend(pred_labels)

all_labels = np.array(all_labels)
all_preds = np.array(all_preds)
all_probs = np.array(all_probs)

# -----------------------------
# 2️⃣ Compute metrics
# -----------------------------
acc = accuracy_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds)
auc = roc_auc_score(all_labels, all_probs)
ap = average_precision_score(all_labels, all_probs)

print(f"Test Accuracy: {acc:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"AUC: {auc:.4f}")
print(f"Average Precision (AP): {ap:.4f}")

# -----------------------------
# 3️⃣ Confusion Matrix
# -----------------------------
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Real','AI'], yticklabels=['Real','AI'])
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.show()

# -----------------------------
# 4️⃣ ROC Curve
# -----------------------------
fpr, tpr, thresholds = roc_curve(all_labels, all_probs)
plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, label=f'AUC = {auc:.3f}')
plt.plot([0,1],[0,1],'k--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()

# -----------------------------
# 5️⃣ Feature Distributions (Statistical Validation)
# -----------------------------
feature_names = [
    "Shannon Entropy","Local Entropy","Gradient Entropy","Laplacian Variance",
    "High-Freq Energy","R_mean","R_std","G_mean","G_std","B_mean","B_std"
]

real_features = features_array[labels_array == 0]
fake_features = features_array[labels_array == 1]

for i, fname in enumerate(feature_names):
    plt.figure(figsize=(6,4))
    sns.kdeplot(real_features[:, i], label="Real", fill=True, alpha=0.5)
    sns.kdeplot(fake_features[:, i], label="AI", fill=True, alpha=0.5)
    plt.title(f"{fname} Distribution: Real vs AI")
    plt.xlabel("Feature Value")
    plt.ylabel("Density")
    plt.legend()
    plt.show()

In [ ]:
import os
print(os.getcwd())

In [ ]:
import os

model_path = os.path.abspath("stat_detector.pth")
csv_path = os.path.abspath("misclassified_test_images.csv")

print(model_path)
print(csv_path)

In [ ]:
import pandas as pd

csv_path = "/content/combined_failure_cases.csv"
df = pd.read_csv(csv_path)

print(df.head())

In [ ]:
correct = 0
total = 0
results = []
failed_files = []  # to track images that could not be read

for _, row in df.iterrows():
    img_path = row["filepath"]    # use full path directly
    label = int(row["label"])     # make sure it's integer

    feat = extract_features(img_path)
    if feat is None:
        failed_files.append(img_path)
        continue  # skip this image; do NOT count it in total

    x = torch.tensor(feat, dtype=torch.float32).unsqueeze(0).to(device)

    with torch.no_grad():
        prob = model(x).item()
        pred = 1 if prob > 0.5 else 0

    if pred == label:
        correct += 1
    total += 1

    results.append([img_path, label, pred, prob])

# Compute accuracy only on successfully read images
accuracy = correct / total if total > 0 else 0
print(f"Evaluated {total} images (successfully read)")
print(f"Correct predictions: {correct}")
print(f"Accuracy: {accuracy:.4f}")

# Save predictions
pred_df = pd.DataFrame(results, columns=["filepath", "true_label", "predicted", "confidence"])
pred_df.to_csv("/content/combined_failure_cases_predictions.csv", index=False)
print("Predictions saved to /content/combined_failure_cases_predictions.csv")

# Optional: print summary of failed files
if failed_files:
    print(f"\nFailed to read {len(failed_files)} images:")
    for f in failed_files[:10]:  # show first 10
        print(f)